
# 02 — Fill non-trained baseline metrics for E-GEO benchmark

This notebook fills the non-trained rows used by Supplementary Table S1. It now computes validation, official-test, and cross-model metrics for Random and TF-IDF cosine baselines, and intentionally skips E5-base official-test scoring, which belongs in the pretrained-neural notebook or can remain as a dash if not evaluated.

**Replace policy:** all CSV outputs are written to the same canonical filenames and are overwritten on rerun.


In [1]:

# Fast non-trained-baseline patch.
# E5-base official-test scoring is intentionally disabled here because it is slow and belongs in notebook 03.
RUN_E5_OFFICIAL_TEST = False
E5_MODEL_NAME = "intfloat/e5-base-v2"
E5_BATCH_SIZE = 64
RANDOM_SEED = 42

# Set this to True only if you intentionally want this notebook to compute the heavy E5 official-test baseline. Otherwise keep it False and let notebook 03 handle pretrained-neural baselines.


In [2]:

from pathlib import Path
import sys, os, math, warnings, subprocess, importlib.util
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Finance Research/E-GEO-ML")
if not PROJECT_ROOT.exists():
    CWD = Path.cwd().resolve()
    if (CWD / "data").exists() or (CWD / "raw_data").exists():
        PROJECT_ROOT = CWD
    elif (CWD.parent / "data").exists() or (CWD.parent / "raw_data").exists():
        PROJECT_ROOT = CWD.parent
    else:
        PROJECT_ROOT = CWD.parent if CWD.name.lower() == "code" else CWD

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLE_DIR = DATA_DIR / "manuscript_tables_benchmark_final"
for d in [DATA_DIR, RESULTS_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [DATA_DIR, PROJECT_ROOT, TABLE_DIR, Path.cwd().resolve()]

def resolve_existing(name_or_path):
    p = Path(name_or_path)
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.append(p)
        for d in SEARCH_DIRS:
            candidates.append(d / p.name)
            if len(p.parts) > 1:
                candidates.append(d / p)
    seen = set()
    for c in candidates:
        c = c.resolve()
        if c in seen:
            continue
        seen.add(c)
        if c.exists():
            return c
    return Path(name_or_path)

def read_csv_optional(name_or_path, **kwargs):
    p = resolve_existing(name_or_path)
    if p.exists():
        print("Loaded:", p)
        return pd.read_csv(p, **kwargs)
    print("Missing:", name_or_path)
    return None

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("TABLE_DIR:", TABLE_DIR)

# ============================================================
# Replace-save policy
# ============================================================
# All generated CSVs in this notebook are written to canonical filenames.
# Existing files with the same names are removed first, then replaced.
# This avoids accumulating large duplicate output files.
# This patch is idempotent, so rerunning this cell in the same kernel is safe.
# ============================================================

if not hasattr(pd.DataFrame, "_egeo_original_to_csv"):
    pd.DataFrame._egeo_original_to_csv = pd.DataFrame.to_csv


def _to_csv_replace(self, path_or_buf=None, *args, **kwargs):
    if isinstance(path_or_buf, (str, os.PathLike, Path)):
        p = Path(path_or_buf)
        p.parent.mkdir(parents=True, exist_ok=True)
        if p.exists():
            p.unlink()
    return pd.DataFrame._egeo_original_to_csv(self, path_or_buf, *args, **kwargs)

pd.DataFrame.to_csv = _to_csv_replace
print("Replace-save policy: enabled. Existing CSV outputs with the same names will be replaced.")




def save_csv_replace(df, path, index=False):
    """Save CSV by replacing the canonical output file instead of creating new versions."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, index=index)
    print("Saved / replaced:", path)
    return path


Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/Finance Research/E-GEO-ML
DATA_DIR: /content/drive/MyDrive/Finance Research/E-GEO-ML/data
TABLE_DIR: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/manuscript_tables_benchmark_final
Replace-save policy: enabled. Existing CSV outputs with the same names will be replaced.


## Helper functions

In [3]:
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize


PROVIDER_STYLE_OUTCOME_COLUMNS = [
    "visibility_presence_proxy",
    "average_position_proxy",
    "rank_score_proxy",
    "share_of_voice_proxy",
    "top3_share_of_voice_proxy",
    "first_position_proxy",
]


def add_provider_style_visibility_metrics(df, group_cols=("split", "model", "query_id")):
    """Add provider-style E-GEO ranking proxy metrics if they are missing."""
    if df is None or len(df) == 0:
        return df
    out = df.copy()
    if "rank" not in out.columns:
        return out

    out["rank"] = pd.to_numeric(out["rank"], errors="coerce")
    if "top1" not in out.columns:
        out["top1"] = (out["rank"] == 1).astype(int)
    if "top3" not in out.columns:
        out["top3"] = (out["rank"] <= 3).astype(int)

    out["top1"] = pd.to_numeric(out["top1"], errors="coerce").fillna(0).astype(int)
    out["top3"] = pd.to_numeric(out["top3"], errors="coerce").fillna(0).astype(int)

    out["visibility_presence_proxy"] = out["top3"].astype(float)
    out["average_position_proxy"] = out["rank"].astype(float)
    out["rank_score_proxy"] = 1.0 / out["rank"].replace(0, np.nan)

    active_group_cols = [c for c in group_cols if c in out.columns]
    if len(active_group_cols) == 0:
        active_group_cols = ["query_id"] if "query_id" in out.columns else []

    if len(active_group_cols) > 0:
        denom = out.groupby(active_group_cols)["rank_score_proxy"].transform("sum")
        out["share_of_voice_proxy"] = out["rank_score_proxy"] / denom.replace(0, np.nan)
        top3_denom = out.groupby(active_group_cols)["top3"].transform("sum")
        out["top3_share_of_voice_proxy"] = out["top3"] / top3_denom.replace(0, np.nan)
    else:
        denom = out["rank_score_proxy"].sum()
        out["share_of_voice_proxy"] = out["rank_score_proxy"] / denom if denom else np.nan
        top3_denom = out["top3"].sum()
        out["top3_share_of_voice_proxy"] = out["top3"] / top3_denom if top3_denom else np.nan

    out["first_position_proxy"] = out["top1"].astype(float)
    return out


def ndcg_at_k_binary(labels, scores, k=3):
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")
    rel = labels[order][:k]
    dcg = sum(rel[i] / math.log2(i + 2) for i in range(len(rel)))
    ideal = np.sort(labels)[::-1][:k]
    idcg = sum(ideal[i] / math.log2(i + 2) for i in range(len(ideal)))
    return dcg / idcg if idcg > 0 else np.nan


def query_level_metrics(eval_df, score_col="score", label_col="top3", k=3):
    eval_df = add_provider_style_visibility_metrics(eval_df)
    rows = []
    for qid, g in eval_df.groupby("query_id"):
        g = g.copy()
        y = pd.to_numeric(g[label_col], errors="coerce").fillna(0).astype(int).to_numpy()
        s = pd.to_numeric(g[score_col], errors="coerce").fillna(-np.inf).to_numpy()
        order = np.argsort(-s, kind="mergesort")
        top_idx = order[:k]
        top = y[top_idx]
        positives = max(int(y.sum()), 1)

        top_g = g.iloc[top_idx]
        out = {
            "query_id": qid,
            "precision@3": float(top.sum() / k),
            "recall@3": float(top.sum() / positives),
            "ndcg@3": float(ndcg_at_k_binary(y, s, k=k)),
            "sov_captured@3": np.nan,
            "top3_sov_captured@3": np.nan,
            "mean_true_position@3": np.nan,
            "top1_hit@3": np.nan,
        }
        if "share_of_voice_proxy" in top_g.columns:
            out["sov_captured@3"] = float(pd.to_numeric(top_g["share_of_voice_proxy"], errors="coerce").fillna(0).sum())
        if "top3_share_of_voice_proxy" in top_g.columns:
            out["top3_sov_captured@3"] = float(pd.to_numeric(top_g["top3_share_of_voice_proxy"], errors="coerce").fillna(0).sum())
        if "rank" in top_g.columns:
            out["mean_true_position@3"] = float(pd.to_numeric(top_g["rank"], errors="coerce").mean())
        if "top1" in top_g.columns:
            out["top1_hit@3"] = float(pd.to_numeric(top_g["top1"], errors="coerce").fillna(0).max() > 0)
        rows.append(out)
    return pd.DataFrame(rows)


def row_metrics(eval_df, score_col="score", label_col="top3"):
    y = pd.to_numeric(eval_df[label_col], errors="coerce").fillna(0).astype(int)
    s = pd.to_numeric(eval_df[score_col], errors="coerce").fillna(0.0)
    out = {
        "AUPRC": float(average_precision_score(y, s)) if y.nunique() > 1 else np.nan,
        "AUROC": float(roc_auc_score(y, s)) if y.nunique() > 1 else np.nan,
    }
    # F1/accuracy are not central for S1, but keeping them makes the file self-contained.
    if s.max() > s.min():
        threshold = float(np.quantile(s, 0.70))  # approximate top-30% positive rate threshold
    else:
        threshold = 0.5
    pred = (s >= threshold).astype(int)
    out.update({
        "Accuracy": float(accuracy_score(y, pred)),
        "F1": float(f1_score(y, pred, zero_division=0)),
        "threshold": threshold,
    })
    return out


def evaluate_scores(df, score_col, method_key, family, representative_method, split, label_model="gpt5", source="computed"):
    if df is None or len(df) == 0:
        return None
    df = add_provider_style_visibility_metrics(df)
    needed = {"query_id", "top3", score_col}
    if not needed.issubset(df.columns):
        missing = sorted(needed - set(df.columns))
        print(f"Skipping {method_key} {split}: missing columns {missing}")
        return None
    q = query_level_metrics(df, score_col=score_col, label_col="top3", k=3)
    r = row_metrics(df, score_col=score_col, label_col="top3")
    out = {
        "family": family,
        "method_key": method_key,
        "representative_method": representative_method,
        "split": split,
        "label_model": label_model,
        "NDCG@3": float(q["ndcg@3"].mean()),
        "precision@3": float(q["precision@3"].mean()),
        "recall@3": float(q["recall@3"].mean()),
        "sov_captured@3": float(q["sov_captured@3"].mean()) if "sov_captured@3" in q.columns else np.nan,
        "top3_sov_captured@3": float(q["top3_sov_captured@3"].mean()) if "top3_sov_captured@3" in q.columns else np.nan,
        "mean_true_position@3": float(q["mean_true_position@3"].mean()) if "mean_true_position@3" in q.columns else np.nan,
        "top1_hit@3": float(q["top1_hit@3"].mean()) if "top1_hit@3" in q.columns else np.nan,
        "n_rows": int(len(df)),
        "n_queries": int(df["query_id"].nunique()),
        "source": source,
    }
    out.update(r)
    return out


def add_tfidf_query_product_cosine(target_df, fit_df=None, out_col="score_tfidf_compact_cosine"):
    df = target_df.copy()
    if out_col in df.columns:
        return df
    if "tfidf_query_product_cosine" in df.columns:
        df[out_col] = pd.to_numeric(df["tfidf_query_product_cosine"], errors="coerce").fillna(0.0)
        return df
    if fit_df is None:
        fit_df = df
    corpus = pd.concat([
        fit_df["query"].fillna("").astype(str),
        fit_df["product_text"].fillna("").astype(str)
    ], ignore_index=True)
    vectorizer = TfidfVectorizer(max_features=60000, ngram_range=(1, 2), min_df=2)
    vectorizer.fit(corpus)
    q = vectorizer.transform(df["query"].fillna("").astype(str))
    p = vectorizer.transform(df["product_text"].fillna("").astype(str))
    q = normalize(q)
    p = normalize(p)
    df[out_col] = np.asarray(q.multiply(p).sum(axis=1)).ravel()
    return df


def add_random_score(df, seed=42, out_col="score_random_ranking"):
    out = df.copy()
    rng = np.random.default_rng(seed)
    # Deterministic but product-row-specific enough for reproducible baseline.
    out[out_col] = rng.random(len(out))
    return out


## Load available processed data

In [4]:

valid_split = read_csv_optional("egeo_gpt5_internal_valid_split.csv")
visibility_gpt5 = read_csv_optional("egeo_train_val_visibility_gpt5.csv")
rankings_long = read_csv_optional("egeo_train_val_rankings_long.csv")
products_flat = read_csv_optional("egeo_train_val_products_flat.csv")

# Official test files are produced by notebook 01 if raw_data/test_data.json and test ranking files were available.
test_gpt5 = read_csv_optional("egeo_test_visibility_gpt5.csv")
test_long = read_csv_optional("egeo_test_visibility_long_all_models.csv")

neural_query_all = read_csv_optional("egeo_pretrained_neural_query_metrics_all_label_models.csv")
neural_row_all = read_csv_optional("egeo_pretrained_neural_row_metrics_all_label_models.csv")

if valid_split is not None:
    print("Validation rows:", valid_split.shape, "queries:", valid_split["query_id"].nunique())
if test_gpt5 is not None:
    print("Official GPT-5 test rows:", test_gpt5.shape, "queries:", test_gpt5["query_id"].nunique())
else:
    print("No egeo_test_visibility_gpt5.csv found. Official-test non-trained baselines cannot be fully computed in this notebook until that file is available.")


Loaded: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_gpt5_internal_valid_split.csv
Loaded: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_train_val_visibility_gpt5.csv
Loaded: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_train_val_rankings_long.csv
Loaded: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_train_val_products_flat.csv
Loaded: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_test_visibility_gpt5.csv
Loaded: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_test_visibility_long_all_models.csv
Loaded: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_pretrained_neural_query_metrics_all_label_models.csv
Loaded: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_pretrained_neural_row_metrics_all_label_models.csv
Validation rows: (3920, 56) queries: 392
Official GPT-5 test rows: (19710, 55) queries: 1971


## Rebuild cross-model validation long table only if needed

In [5]:

# For random cross-model validation, try to use egeo_train_val_visibility_long_all_models.csv.
# If it was not saved/uploaded, rebuild the minimal long table by merging product rows with ranking labels.
train_val_long = read_csv_optional("egeo_train_val_visibility_long_all_models.csv")
if train_val_long is None and products_flat is not None and rankings_long is not None:
    print("Rebuilding minimal train_val long table from products_flat + rankings_long...")
    key_cols = [c for c in ["query_id", "custom_id", "product_num", "product_id"] if c in products_flat.columns and c in rankings_long.columns]
    if len(key_cols) >= 2:
        train_val_long = products_flat.merge(rankings_long, on=key_cols, how="inner", suffixes=("", "_rank"))
        print("Rebuilt train_val_long:", train_val_long.shape)
    else:
        print("Could not rebuild long table: shared key columns are insufficient:", key_cols)


Loaded: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_train_val_visibility_long_all_models.csv


## Compute fast random and lexical baselines

In [6]:

rows = []

# Validation random baseline for GPT-5
if valid_split is not None:
    tmp = add_random_score(valid_split, seed=RANDOM_SEED)
    rows.append(evaluate_scores(
        tmp, "score_random_ranking", "random_ranking", "Random", "Random ranking", "validation", "gpt5", "02_computed"
    ))

# Cross-model random baseline on train/validation labels
if train_val_long is not None and {"model", "query_id", "top3"}.issubset(train_val_long.columns):
    for label_model, d in train_val_long.groupby("model"):
        d = add_random_score(d, seed=RANDOM_SEED)
        rows.append(evaluate_scores(
            d, "score_random_ranking", "random_ranking", "Random", "Random ranking", "validation", str(label_model), "02_computed_cross_model"
        ))

# Official test random baseline for GPT-5
if test_gpt5 is not None:
    tmp = add_random_score(test_gpt5, seed=RANDOM_SEED)
    rows.append(evaluate_scores(
        tmp, "score_random_ranking", "random_ranking", "Random", "Random ranking", "official_test", "gpt5", "02_computed"
    ))

# Validation lexical baseline.
# If 03 has already been rerun with provider-style metrics, we can reuse its table.
# Otherwise, recompute TF-IDF here so provider-style metrics are populated immediately.
used_precomputed_tfidf = False
if neural_query_all is not None and neural_row_all is not None:
    q = neural_query_all[(neural_query_all["label_model"] == "gpt5") & (neural_query_all["score_model"] == "tfidf_compact_cosine")]
    r = neural_row_all[(neural_row_all["label_model"] == "gpt5") & (neural_row_all["score_model"] == "tfidf_compact_cosine")]
    provider_metric_cols = {"sov_captured@3", "top3_sov_captured@3", "mean_true_position@3", "top1_hit@3"}
    if len(q) > 0 and provider_metric_cols.issubset(set(q.columns)):
        rows.append({
            "family": "Lexical", "method_key": "tfidf_compact_cosine", "representative_method": "TF-IDF cosine",
            "split": "validation", "label_model": "gpt5",
            "NDCG@3": float(q.iloc[0]["ndcg@3"]),
            "precision@3": float(q.iloc[0].get("precision@3", np.nan)),
            "recall@3": float(q.iloc[0].get("recall@3", np.nan)),
            "sov_captured@3": float(q.iloc[0].get("sov_captured@3", np.nan)),
            "top3_sov_captured@3": float(q.iloc[0].get("top3_sov_captured@3", np.nan)),
            "mean_true_position@3": float(q.iloc[0].get("mean_true_position@3", np.nan)),
            "top1_hit@3": float(q.iloc[0].get("top1_hit@3", np.nan)),
            "AUPRC": float(r.iloc[0]["AUPRC"]) if len(r) > 0 and "AUPRC" in r.columns else np.nan,
            "AUROC": float(r.iloc[0]["AUROC"]) if len(r) > 0 and "AUROC" in r.columns else np.nan,
            "Accuracy": float(r.iloc[0]["Accuracy"]) if len(r) > 0 and "Accuracy" in r.columns else np.nan,
            "F1": float(r.iloc[0]["F1"]) if len(r) > 0 and "F1" in r.columns else np.nan,
            "threshold": float(r.iloc[0]["threshold"]) if len(r) > 0 and "threshold" in r.columns else np.nan,
            "n_rows": int(q.iloc[0].get("n_rows", np.nan)) if pd.notna(q.iloc[0].get("n_rows", np.nan)) else np.nan,
            "n_queries": int(q.iloc[0].get("n_queries", np.nan)) if pd.notna(q.iloc[0].get("n_queries", np.nan)) else np.nan,
            "source": "precomputed_neural_table_with_provider_metrics",
        })
        used_precomputed_tfidf = True

if not used_precomputed_tfidf and valid_split is not None:
    tmp = add_tfidf_query_product_cosine(valid_split, fit_df=visibility_gpt5 if visibility_gpt5 is not None else valid_split)
    rows.append(evaluate_scores(
        tmp, "score_tfidf_compact_cosine", "tfidf_compact_cosine", "Lexical", "TF-IDF cosine", "validation", "gpt5", "02_computed_provider_metrics"
    ))

# Cross-model lexical/pretrained rows from precomputed neural table.
if neural_query_all is not None and neural_row_all is not None:
    merged = neural_query_all.merge(
        neural_row_all,
        on=["label_model", "score_model", "score_col", "n_rows", "n_queries"],
        how="left",
        suffixes=("_query", "_row")
    )
    for _, x in merged.iterrows():
        fam = "Lexical" if x["score_model"] == "tfidf_compact_cosine" else "Pretrained neural"
        rep = "TF-IDF cosine" if x["score_model"] == "tfidf_compact_cosine" else str(x["score_model"]).replace("embed_e5_base_cosine", "E5-base").replace("_", " ")
        rows.append({
            "family": fam,
            "method_key": x["score_model"],
            "representative_method": rep,
            "split": "validation",
            "label_model": x["label_model"],
            "NDCG@3": float(x["ndcg@3"]),
            "precision@3": float(x.get("precision@3", np.nan)),
            "recall@3": float(x.get("recall@3", np.nan)),
            "sov_captured@3": float(x.get("sov_captured@3", np.nan)),
            "top3_sov_captured@3": float(x.get("top3_sov_captured@3", np.nan)),
            "mean_true_position@3": float(x.get("mean_true_position@3", np.nan)),
            "top1_hit@3": float(x.get("top1_hit@3", np.nan)),
            "AUPRC": float(x.get("AUPRC", np.nan)),
            "AUROC": float(x.get("AUROC", np.nan)),
            "Accuracy": float(x.get("Accuracy", np.nan)),
            "F1": float(x.get("F1", np.nan)),
            "threshold": float(x.get("threshold", np.nan)),
            "n_rows": int(x.get("n_rows", np.nan)) if pd.notna(x.get("n_rows", np.nan)) else np.nan,
            "n_queries": int(x.get("n_queries", np.nan)) if pd.notna(x.get("n_queries", np.nan)) else np.nan,
            "source": "precomputed_neural_table",
        })

# Official test lexical baseline
if test_gpt5 is not None:
    fit_for_tfidf = visibility_gpt5 if visibility_gpt5 is not None else valid_split
    tmp = add_tfidf_query_product_cosine(test_gpt5, fit_df=fit_for_tfidf)
    rows.append(evaluate_scores(
        tmp, "score_tfidf_compact_cosine", "tfidf_compact_cosine", "Lexical", "TF-IDF cosine", "official_test", "gpt5", "02_computed"
    ))

rows = [r for r in rows if r is not None]
summary = pd.DataFrame(rows)
summary = summary.drop_duplicates(subset=["method_key", "split", "label_model"], keep="first")
print(summary.shape)
display(summary.head(20))


(56, 20)


,family,method_key,representative_method,split,label_model,NDCG@3,precision@3,recall@3,sov_captured@3,top3_sov_captured@3,mean_true_position@3,top1_hit@3,n_rows,n_queries,source,AUPRC,AUROC,Accuracy,F1,threshold
0,Random,random_ranking,Random ranking,validation,gpt5,0.325690,0.323129,0.323129,0.312236,0.323129,5.335884,0.321429,3920,392,02_computed,0.310188,0.511051,0.582143,0.303571,0.702312
1,Random,random_ranking,Random ranking,validation,claude,0.309221,0.307333,0.307333,0.305280,0.307333,5.440833,0.313000,20000,2000,02_computed_cross_model,0.304913,0.505258,0.583200,0.305333,0.699930
2,Random,random_ranking,Random ranking,validation,deepseek,0.306675,0.301726,0.306509,0.303983,0.306509,5.481292,0.296771,19383,1951,02_computed_cross_model,0.302723,0.507288,0.581437,0.300543,0.699214
3,Random,random_ranking,Random ranking,validation,gemini,0.300628,0.299167,0.299167,0.298032,0.299167,5.493333,0.291500,20000,2000,02_computed_cross_model,0.299664,0.496514,0.578100,0.296833,0.699930
4,Random,random_ranking,Random ranking,validation,gpt41,0.302131,0.303167,0.303167,0.301533,0.303167,5.492333,0.305000,19999,2000,02_computed_cross_model,0.298247,0.498403,0.582329,0.303859,0.699937
6,Random,random_ranking,Random ranking,validation,llama,0.309194,0.306573,0.306573,0.300787,0.306573,5.508948,0.303562,19918,1993,02_computed_cross_model,0.304356,0.503731,0.583342,0.305813,0.699908
7,Random,random_ranking,Random ranking,official_test,gpt5,0.302898,0.301032,0.301032,0.303585,0.301032,5.486724,0.314054,19710,1971,02_computed,0.300426,0.499866,0.576763,0.294605,0.699674
8,Lexical,tfidf_compact_cosine,TF-IDF cosine,validation,gpt5,0.293909,0.290816,0.290816,0.296045,0.290816,5.524660,0.293367,3920,392,02_computed_provider_metrics,0.295470,0.495163,0.579592,0.299320,0.202200
9,Pretrained neural,hybrid_all_available,hybrid all available,validation,claude,0.362231,0.352041,0.352041,NaN,NaN,NaN,NaN,3920,392,precomputed_neural_table,0.340744,0.552078,0.610204,0.350340,0.611248
10,Pretrained neural,rerank_msmarco_minilm_l6,rerank msmarco minilm l6,validation,claude,0.359466,0.357993,0.357993,NaN,NaN,NaN,NaN,3920,392,precomputed_neural_table,0.316570,0.526914,0.596939,0.328231,0.398409


## Optional: compute E5-base baseline on official test

In [7]:
# ============================================================
# E5-base official-test scoring is intentionally NOT computed in notebook 02.
#
# This notebook should stay fast and only fill Random + TF-IDF cosine
# official-test / cross-model baseline values.
#
# If you want to fill the E5-base official-test dashes in Table S1,
# run notebook 03 instead. The heavy SentenceTransformer encoding belongs there.
# ============================================================
print("Skipping E5 official-test scoring in notebook 02. Run notebook 03 for E5-base official-test metrics.")


Skipping E5 official-test scoring in notebook 02. Run notebook 03 for E5-base official-test metrics.


## Save 02 outputs

In [8]:

# Standardize method labels.
METHOD_LABEL = {
    "random_ranking": "Random ranking",
    "tfidf_compact_cosine": "TF-IDF cosine",
    "embed_e5_base_cosine": "E5-base",
    "embed_minilm_l6_cosine": "MiniLM-L6 embedding",
    "embed_multiqa_minilm_cosine": "MultiQA MiniLM embedding",
    "rerank_msmarco_minilm_l6": "MS MARCO MiniLM-L6 reranker",
    "rerank_msmarco_minilm_l12": "MS MARCO MiniLM-L12 reranker",
    "rerank_bge_base": "BGE-base reranker",
    "hybrid_all_available": "Hybrid pretrained ensemble",
}
if len(summary) > 0:
    summary["representative_method"] = summary["method_key"].map(METHOD_LABEL).fillna(summary["representative_method"])
    sort_cols = ["family", "method_key", "split", "label_model"]
    summary = summary.sort_values(sort_cols).reset_index(drop=True)

summary_path = DATA_DIR / "egeo_02_nontrained_baseline_summary.csv"
save_csv_replace(summary, summary_path, index=False)

gpt5_view = summary[summary["label_model"].astype(str).eq("gpt5")].copy()
gpt5_path = DATA_DIR / "egeo_02_nontrained_baseline_gpt5_view.csv"
save_csv_replace(gpt5_view, gpt5_path, index=False)

cross_model_view = (
    summary[summary["split"].eq("validation")]
    .groupby(["family", "method_key", "representative_method"], as_index=False)
    .agg(
        cross_model_mean_ndcg3=("NDCG@3", "mean"),
        cross_model_mean_sov_captured3=("sov_captured@3", "mean"),
        cross_model_mean_top1_hit3=("top1_hit@3", "mean"),
        cross_model_mean_true_position3=("mean_true_position@3", "mean"),
        n_label_models=("label_model", "nunique")
    )
)
cross_path = DATA_DIR / "egeo_02_nontrained_cross_model_means.csv"
save_csv_replace(cross_model_view, cross_path, index=False)


# Additional canonical aliases used by some reporting notebooks. These are replaced, not versioned.
save_csv_replace(summary, DATA_DIR / "egeo_nontrained_baseline_metrics.csv", index=False)
save_csv_replace(cross_model_view, DATA_DIR / "egeo_nontrained_cross_model_means.csv", index=False)

display_cols = ["family", "method_key", "split", "label_model", "NDCG@3", "sov_captured@3", "top1_hit@3", "mean_true_position@3", "AUPRC", "source"]
display_cols = [c for c in display_cols if c in gpt5_view.columns]
display(gpt5_view[display_cols].sort_values(["family", "method_key", "split"]))
display(cross_model_view)


Saved / replaced: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_02_nontrained_baseline_summary.csv
Saved / replaced: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_02_nontrained_baseline_gpt5_view.csv
Saved / replaced: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_02_nontrained_cross_model_means.csv
Saved / replaced: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_nontrained_baseline_metrics.csv
Saved / replaced: /content/drive/MyDrive/Finance Research/E-GEO-ML/data/egeo_nontrained_cross_model_means.csv


,family,method_key,split,label_model,NDCG@3,sov_captured@3,top1_hit@3,mean_true_position@3,AUPRC,source
0,Lexical,tfidf_compact_cosine,official_test,gpt5,0.315565,0.313145,0.348554,5.408929,0.304039,02_computed
5,Lexical,tfidf_compact_cosine,validation,gpt5,0.293909,0.296045,0.293367,5.524660,0.295470,02_computed_provider_metrics
11,Pretrained neural,embed_e5_base_cosine,validation,gpt5,0.352369,NaN,NaN,NaN,0.319822,precomputed_neural_table
17,Pretrained neural,embed_minilm_l6_cosine,validation,gpt5,0.270593,NaN,NaN,NaN,0.295875,precomputed_neural_table
23,Pretrained neural,embed_multiqa_minilm_cosine,validation,gpt5,0.302659,NaN,NaN,NaN,0.301795,precomputed_neural_table
29,Pretrained neural,hybrid_all_available,validation,gpt5,0.319519,NaN,NaN,NaN,0.323198,precomputed_neural_table
35,Pretrained neural,rerank_bge_base,validation,gpt5,0.323281,NaN,NaN,NaN,0.312599,precomputed_neural_table
41,Pretrained neural,rerank_msmarco_minilm_l12,validation,gpt5,0.333500,NaN,NaN,NaN,0.309869,precomputed_neural_table
47,Pretrained neural,rerank_msmarco_minilm_l6,validation,gpt5,0.338587,NaN,NaN,NaN,0.310342,precomputed_neural_table
49,Random,random_ranking,official_test,gpt5,0.302898,0.303585,0.314054,5.486724,0.300426,02_computed


,family,method_key,representative_method,cross_model_mean_ndcg3,cross_model_mean_sov_captured3,cross_model_mean_top1_hit3,cross_model_mean_true_position3,n_label_models
0,Lexical,tfidf_compact_cosine,TF-IDF cosine,0.324812,0.296045,0.293367,5.524660,6
1,Pretrained neural,embed_e5_base_cosine,E5-base,0.355606,NaN,NaN,NaN,6
2,Pretrained neural,embed_minilm_l6_cosine,MiniLM-L6 embedding,0.320892,NaN,NaN,NaN,6
3,Pretrained neural,embed_multiqa_minilm_cosine,MultiQA MiniLM embedding,0.310882,NaN,NaN,NaN,6
4,Pretrained neural,hybrid_all_available,Hybrid pretrained ensemble,0.359658,NaN,NaN,NaN,6
5,Pretrained neural,rerank_bge_base,BGE-base reranker,0.349552,NaN,NaN,NaN,6
6,Pretrained neural,rerank_msmarco_minilm_l12,MS MARCO MiniLM-L12 reranker,0.344995,NaN,NaN,NaN,6
7,Pretrained neural,rerank_msmarco_minilm_l6,MS MARCO MiniLM-L6 reranker,0.359361,NaN,NaN,NaN,6
8,Random,random_ranking,Random ranking,0.308923,0.303642,0.305210,5.458771,6
